In [1]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict
import time

In [2]:
class CrashState(TypedDict):
    input: str
    step1: str
    step2: str

In [3]:
def step_1(state: CrashState) -> CrashState:
    print("Step 1 executed")
    return {"step1": "done", "input": state["input"]}

def step_2(state: CrashState) -> CrashState:
    print("Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)")

def step_3(state: CrashState) -> CrashState:
    print("Step 3 executed")
    return {"done": True}

In [7]:
builder = StateGraph(CrashState)
builder.add_node("step_1", step_1)
builder.add_node("step_2", step_2)
builder.add_node("step_3", step_3)

builder.set_entry_point("step_1")
builder.add_edge("step_1", "step_2")
builder.add_edge("step_2", "step_3")
builder.add_edge("step_3", END)

checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

In [8]:
try:
    print("Running graph: Please manually interrupt during Step 2...")
    graph.invoke({"input": "start"}, config={"configurable": {"thread_id": 'thread-1'}})
except KeyboardInterrupt:
    print("Kernel manually interrupted (crash simulated).")

Running graph: Please manually interrupt during Step 2...
Step 1 executed
Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)
Step 3 executed


In [9]:
# 6. Re-run to show fault-tolerant resume
print("\nRe-running the graph to demonstrate fault tolerance...")
final_state = graph.invoke(None, config={"configurable": {"thread_id": 'thread-1'}})
print("\nFinal State:", final_state)


Re-running the graph to demonstrate fault tolerance...

Final State: {'input': 'start', 'step1': 'done'}


In [10]:
list(graph.get_state_history({"configurable": {"thread_id": 'thread-1'}}))

[StateSnapshot(values={'input': 'start', 'step1': 'done'}, next=(), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f16fed0-ef3c-6718-8003-dc4ca7b0ca51'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}}, created_at='2026-06-24T16:52:17.398551+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f16fed0-ef3b-6d04-8002-41c074768e41'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'input': 'start', 'step1': 'done'}, next=('step_3',), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f16fed0-ef3b-6d04-8002-41c074768e41'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-06-24T16:52:17.398291+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f16fed0-ef3a-6ad0-8001-f256062d4a2b'}}, tasks=(PregelTask(id='444e1493-f0e7-c49a-a95f-a0aaf5a1fb30', name='step_3', path=('__pregel